# Campaign Intelligence System - working prototype
**DSE3170 Project Based Learning 3**

Two logistic regression models (no tree models):
1. **Fatigue model (main)** - probability that the next call is a brush-off, shown as a score from 0 to 100.
2. **Propensity model (supporting)** - probability that the customer subscribes.

A decision matrix turns the two scores into one action: *contact now, wait then retry, low priority* or *suppress*.

**How to run:** Runtime -> Run all. Colab already has every library this notebook needs.

## 1. Get the real data (UCI Bank Marketing, 41,188 calls)

In [ ]:
import os, zipfile, urllib.request, io
os.makedirs("data", exist_ok=True); os.makedirs("outputs", exist_ok=True)
URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
try:
    outer = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL, timeout=60).read()))
    inner = zipfile.ZipFile(io.BytesIO(outer.read("bank-additional.zip")))
    open("data/bank_marketing.csv", "wb").write(inner.read("bank-additional/bank-additional-full.csv"))
    print("Downloaded data/bank_marketing.csv")
except Exception as e:
    print("Download failed:", e)
    print("Upload bank_marketing.csv from the project zip's data/ folder instead:")
    from google.colab import files
    up = files.upload()
    open("data/bank_marketing.csv", "wb").write(next(iter(up.values())))

## 2. Model code (`cis_core.py`)
Data loading, the brush-off label, both regressions, scoring and the decision matrix.

In [ ]:
%%writefile cis_core.py
"""
Campaign Intelligence System (CIS) - core logic.

Two logistic regression models, no tree models:
  1. Fatigue model (main)        - how likely is the customer to brush off the next call?
  2. Propensity model (support)  - how likely is the customer to subscribe?
A decision matrix then combines the two scores into one action.

Data: UCI Bank Marketing (Moro, Cortez & Rita, 2014), 41,188 phone contacts.
"""
import json
import math

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42

# ---------------------------------------------------------------- settings
# A "brush-off" is a call that lasted under 60 seconds and did not lead to a sale.
BRUSH_OFF_SECONDS = 60

# Fatigue model inputs: the customer's contact history only.
FATIGUE_FEATURES = ["calls_this_campaign", "earlier_contacts", "landline"]

# Propensity model inputs: who the customer is and the market at the time.
# Call duration is NOT used because it is only known after the call (leakage).
PROP_NUMERIC = ["age", "euribor3m"]
PROP_CATEGORICAL = ["job", "marital", "education", "default", "poutcome", "month", "contact"]

# Score bands (same as the report): 0-30 Low, 31-60 Medium, 61-100 High.
LOW_MAX, MEDIUM_MAX = 30, 60


# ---------------------------------------------------------------- data
def load_data(path="data/bank_marketing.csv"):
    """Read the raw UCI file and add the columns the models need."""
    df = pd.read_csv(path, sep=";")
    df.columns = [c.replace(".", "_") for c in df.columns]

    df["subscribed"] = (df["y"] == "yes").astype(int)
    df["calls_this_campaign"] = df["campaign"]          # includes the current call
    df["earlier_contacts"] = df["previous"]             # contacts in earlier campaigns
    df["landline"] = (df["contact"] == "telephone").astype(int)
    df["brush_off"] = ((df["duration"] < BRUSH_OFF_SECONDS) & (df["subscribed"] == 0)).astype(int)
    return df


def split(df):
    """80/20 split, stratified on the outcome, fixed seed so results repeat."""
    return train_test_split(df, test_size=0.2, random_state=RANDOM_STATE,
                            stratify=df["subscribed"])


# ---------------------------------------------------------------- models
def train_fatigue_model(train):
    """Plain logistic regression on raw (unscaled) counts, so each weight reads
    directly as 'change in log-odds per extra call / contact'."""
    model = LogisticRegression(max_iter=1000)
    model.fit(train[FATIGUE_FEATURES], train["brush_off"])
    return model


def train_propensity_model(train):
    """Logistic regression with one-hot encoded categories and scaled numbers."""
    prep = ColumnTransformer([
        ("num", StandardScaler(), PROP_NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore"), PROP_CATEGORICAL),
    ])
    model = Pipeline([("prep", prep), ("lr", LogisticRegression(max_iter=2000, C=1.0))])
    model.fit(train[PROP_NUMERIC + PROP_CATEGORICAL], train["subscribed"])
    return model


# ---------------------------------------------------------------- scoring
def fatigue_score(model, X):
    """Fatigue score = P(brush-off) x 100, i.e. 1 / (1 + e^-z) x 100."""
    return model.predict_proba(X[FATIGUE_FEATURES])[:, 1] * 100


def propensity_score(model, X):
    return model.predict_proba(X[PROP_NUMERIC + PROP_CATEGORICAL])[:, 1]


def fatigue_band(score):
    if score <= LOW_MAX:
        return "Low"
    if score <= MEDIUM_MAX:
        return "Medium"
    return "High"


def decide(propensity, fatigue, high_prop_cut, opted_out=False):
    """Decision matrix: propensity (high/low) x fatigue (rested/tired)."""
    if opted_out:
        return "DO NOT CONTACT"
    high = propensity >= high_prop_cut
    tired = fatigue > LOW_MAX
    if high and not tired:
        return "CONTACT NOW"
    if high and tired:
        return "WAIT, THEN RETRY"
    if tired:
        return "SUPPRESS"
    return "LOW PRIORITY"


# ---------------------------------------------------------------- export
def export_params(fat_model, prop_model, high_prop_cut, metrics, path="outputs/model_params.json"):
    """Write every learned number to JSON, so the web demo can score customers
    with exactly the same maths as Python (no server needed)."""
    prep = prop_model.named_steps["prep"]
    lr = prop_model.named_steps["lr"]
    scaler = prep.named_transformers_["num"]
    encoder = prep.named_transformers_["cat"]
    coefs = lr.coef_[0]

    numeric = {}
    for i, name in enumerate(PROP_NUMERIC):
        numeric[name] = {"mean": float(scaler.mean_[i]), "std": float(scaler.scale_[i]),
                         "coef": float(coefs[i])}
    categorical, k = {}, len(PROP_NUMERIC)
    for name, levels in zip(PROP_CATEGORICAL, encoder.categories_):
        categorical[name] = {}
        for level in levels:
            categorical[name][str(level)] = float(coefs[k])
            k += 1

    params = {
        "fatigue": {
            "intercept": float(fat_model.intercept_[0]),
            "coef": {f: float(c) for f, c in zip(FATIGUE_FEATURES, fat_model.coef_[0])},
            "bands": {"low_max": LOW_MAX, "medium_max": MEDIUM_MAX},
        },
        "propensity": {
            "intercept": float(lr.intercept_[0]),
            "numeric": numeric,
            "categorical": categorical,
            "high_cut": float(high_prop_cut),
        },
        "metrics": metrics,
    }
    with open(path, "w") as f:
        json.dump(params, f, indent=2)
    return params


def sigmoid(z):
    return 1 / (1 + math.exp(-z))


def score_from_params(params, customer):
    """Score one customer using only the saved JSON numbers (what app.py and the
    web demo do). customer is a dict with the propensity fields plus
    calls_this_campaign, earlier_contacts and contact."""
    f = params["fatigue"]
    landline = 1 if customer["contact"] == "telephone" else 0
    z_fat = (f["intercept"]
             + f["coef"]["calls_this_campaign"] * customer["calls_this_campaign"]
             + f["coef"]["earlier_contacts"] * customer["earlier_contacts"]
             + f["coef"]["landline"] * landline)
    fatigue = sigmoid(z_fat) * 100

    p = params["propensity"]
    z_prop = p["intercept"]
    for name, n in p["numeric"].items():
        z_prop += n["coef"] * (customer[name] - n["mean"]) / n["std"]
    for name, levels in p["categorical"].items():
        z_prop += levels.get(str(customer[name]), 0.0)
    propensity = sigmoid(z_prop)

    action = decide(propensity, fatigue, p["high_cut"], customer.get("opted_out", False))
    return {"fatigue_z": z_fat, "fatigue": fatigue, "band": fatigue_band(fatigue),
            "propensity": propensity, "action": action}


# Six worked examples used by score_demo.py, the Streamlit app and the web demo.
# E and F are the same person: F is E after 15 calls.
EXAMPLES = {
    "A - past buyer, first call": dict(age=35, job="admin.", marital="single", education="university.degree",
        default="no", poutcome="success", month="oct", contact="cellular", euribor3m=1.0,
        calls_this_campaign=1, earlier_contacts=2),
    "B - average customer": dict(age=45, job="blue-collar", marital="married", education="basic.9y",
        default="unknown", poutcome="nonexistent", month="may", contact="telephone", euribor3m=4.86,
        calls_this_campaign=3, earlier_contacts=0),
    "C - called 10 times on landline": dict(age=50, job="technician", marital="married", education="professional.course",
        default="no", poutcome="nonexistent", month="may", contact="telephone", euribor3m=4.96,
        calls_this_campaign=10, earlier_contacts=0),
    "D - good prospect, over-called": dict(age=29, job="student", marital="single", education="high.school",
        default="no", poutcome="failure", month="apr", contact="cellular", euribor3m=1.4,
        calls_this_campaign=12, earlier_contacts=0),
    "E - retiree, first call": dict(age=62, job="retired", marital="married", education="basic.4y",
        default="no", poutcome="nonexistent", month="mar", contact="cellular", euribor3m=0.9,
        calls_this_campaign=1, earlier_contacts=0),
    "F - same retiree after 15 calls": dict(age=62, job="retired", marital="married", education="basic.4y",
        default="no", poutcome="nonexistent", month="mar", contact="cellular", euribor3m=0.9,
        calls_this_campaign=15, earlier_contacts=0),
}

## 3. Train both models and print the results (`train.py`)

In [ ]:
%%writefile train.py
"""
Train both regression models, print the results and save charts + model_params.json.
Run:  python train.py
"""
import json
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss, roc_auc_score, roc_curve

import cis_core as cis

os.makedirs("outputs", exist_ok=True)
plt.rcParams.update({"font.family": "Arial", "font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold"})
TEAL, ORANGE, GREY, RED = "#0F766E", "#EA580C", "#94A3B8", "#B91C1C"

# ---------------------------------------------------------------- 1. data
df = cis.load_data()
train, test = cis.split(df)
print(f"Rows: {len(df):,}  (train {len(train):,} / test {len(test):,})")
print(f"Brush-off rate: {df.brush_off.mean():.1%}   Subscription rate: {df.subscribed.mean():.1%}")

# ---------------------------------------------------------------- 2. fatigue model (main)
fat = cis.train_fatigue_model(train)
test = test.copy()
test["fatigue"] = cis.fatigue_score(fat, test)
test["band"] = test["fatigue"].apply(cis.fatigue_band)
fat_auc = roc_auc_score(test.brush_off, test.fatigue)
fat_brier = brier_score_loss(test.brush_off, test.fatigue / 100)

print("\n=== FATIGUE MODEL (main) ===")
print(f"z = {fat.intercept_[0]:+.3f}", end="")
for f, c in zip(cis.FATIGUE_FEATURES, fat.coef_[0]):
    print(f"  {c:+.3f} x {f}", end="")
print()
for f, c in zip(cis.FATIGUE_FEATURES, fat.coef_[0]):
    print(f"  {f:22s} weight {c:+.3f}   odds x {np.exp(c):.2f}")
print(f"AUC {fat_auc:.3f}   Brier {fat_brier:.3f}")

band_table = (test.groupby("band")
              .agg(customers=("brush_off", "size"), predicted=("fatigue", "mean"),
                   actual_brush_off=("brush_off", "mean"))
              .reindex(["Low", "Medium", "High"]))
band_table["share"] = band_table.customers / len(test)
print(band_table.round(3))

# how many customers get flagged at other cut-offs (for the viva question "why so few?")
flag_table = {cut: float((test.fatigue > cut).mean()) for cut in (15, 20, 25, 30)}
print("Share flagged as tired at cut-off:", {k: f"{v:.1%}" for k, v in flag_table.items()})

# ---------------------------------------------------------------- 3. propensity model (support)
prop = cis.train_propensity_model(train)
test["propensity"] = cis.propensity_score(prop, test)
train_prop = cis.propensity_score(prop, train)
high_cut = float(np.quantile(train_prop, 0.70))       # top 30% = "high propensity"
prop_auc = roc_auc_score(test.subscribed, test.propensity)
test["decile"] = pd.qcut(test.propensity.rank(method="first", ascending=False), 10, labels=range(1, 11))
lift = test.groupby("decile", observed=True).subscribed.mean()
base = test.subscribed.mean()

print("\n=== PROPENSITY MODEL (supporting evidence) ===")
print(f"AUC {prop_auc:.3f}   base rate {base:.1%}   top-10% rate {lift.iloc[0]:.1%}  "
      f"(lift {lift.iloc[0] / base:.1f}x)   high-propensity cut {high_cut:.3f}")

# ---------------------------------------------------------------- 4. decision matrix
test["action"] = [cis.decide(p, f, high_cut) for p, f in zip(test.propensity, test.fatigue)]
order = ["CONTACT NOW", "WAIT, THEN RETRY", "LOW PRIORITY", "SUPPRESS"]
decision = (test.groupby("action")
            .agg(customers=("subscribed", "size"), subscribe_rate=("subscribed", "mean"),
                 brush_off_rate=("brush_off", "mean"))
            .reindex(order))
print("\n=== DECISION MATRIX on held-out customers ===")
print(decision.round(3))

# ---------------------------------------------------------------- 5. charts
# 5a. brush-off rate vs number of calls: data vs model
g = test.assign(calls=test.calls_this_campaign.clip(upper=15)).groupby("calls")
actual = g.brush_off.mean()
grid = pd.DataFrame({"calls_this_campaign": range(1, 16), "earlier_contacts": 0, "landline": 0})
grid_l = grid.assign(landline=1)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(actual.index, actual.values * 100, color=GREY, label="Actual brush-off rate (test data)")
ax.plot(grid.calls_this_campaign, cis.fatigue_score(fat, grid), color=TEAL, lw=2.5, marker="o",
        label="Fatigue score - mobile")
ax.plot(grid.calls_this_campaign, cis.fatigue_score(fat, grid_l), color=ORANGE, lw=2.5, marker="o",
        label="Fatigue score - landline")
ax.axhline(30, color=RED, ls="--", lw=1)
ax.text(1, 31.5, "Medium band starts (30)", color=RED, fontsize=9)
ax.set_xlabel("Calls made to the customer in this campaign (15 = 15 or more)")
ax.set_ylabel("Brush-off rate / fatigue score (%)")
ax.set_title("Fatigue rises with every extra call")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout(); fig.savefig("outputs/fatigue_curve.png", dpi=200); plt.close(fig)

# 5b. ROC curves
fig, ax = plt.subplots(figsize=(5, 4.5))
for y, s, name, col, auc in [(test.brush_off, test.fatigue, "Fatigue model", TEAL, fat_auc),
                             (test.subscribed, test.propensity, "Propensity model", ORANGE, prop_auc)]:
    fpr, tpr, _ = roc_curve(y, s)
    ax.plot(fpr, tpr, color=col, lw=2.5, label=f"{name} (AUC {auc:.2f})")
ax.plot([0, 1], [0, 1], color=GREY, ls="--", lw=1, label="Random guess (AUC 0.50)")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("ROC curves on held-out customers")
ax.legend(frameon=False, fontsize=9, loc="lower right")
fig.tight_layout(); fig.savefig("outputs/roc_curves.png", dpi=200); plt.close(fig)

# 5c. lift by propensity decile
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(lift.index.astype(int), lift.values * 100, color=[ORANGE] + [GREY] * 9)
ax.axhline(base * 100, color=TEAL, ls="--", lw=1.5)
ax.text(10.4, base * 100 + 1, f"Average {base:.1%}", color=TEAL, ha="right", fontsize=9)
for b, v in zip(bars, lift.values):
    ax.text(b.get_x() + b.get_width() / 2, v * 100 + 0.8, f"{v:.0%}", ha="center", fontsize=9)
ax.set_xticks(range(1, 11)); ax.set_xlabel("Propensity decile (1 = top 10% of scores)")
ax.set_ylabel("Actual subscription rate (%)")
ax.set_title("Propensity model ranks customers well")
fig.tight_layout(); fig.savefig("outputs/propensity_lift.png", dpi=200); plt.close(fig)

# 5d. outcomes by decision
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(order)); w = 0.38
ax.bar(x - w / 2, decision.subscribe_rate * 100, w, color=TEAL, label="Subscribed")
ax.bar(x + w / 2, decision.brush_off_rate * 100, w, color=ORANGE, label="Brushed off")
for i, (s, b) in enumerate(zip(decision.subscribe_rate, decision.brush_off_rate)):
    ax.text(i - w / 2, s * 100 + 1, f"{s:.0%}", ha="center", fontsize=9)
    ax.text(i + w / 2, b * 100 + 1, f"{b:.0%}", ha="center", fontsize=9)
ax.set_xticks(x, [f"{a}\n(n={n:,})" for a, n in zip(order, decision.customers)], fontsize=9)
ax.set_ylabel("Rate on held-out customers (%)")
ax.set_title("What happened to customers in each decision group")
ax.legend(frameon=False)
fig.tight_layout(); fig.savefig("outputs/decision_outcomes.png", dpi=200); plt.close(fig)

# ---------------------------------------------------------------- 6. save everything
metrics = {
    "rows": len(df), "train_rows": len(train), "test_rows": len(test),
    "brush_off_rate": float(df.brush_off.mean()), "subscribe_rate": float(df.subscribed.mean()),
    "brush_off_by_calls": {int(k): float(v) for k, v in
                           df.groupby(df.campaign.clip(upper=15)).brush_off.mean().items()},
    "fatigue_auc": float(fat_auc), "fatigue_brier": float(fat_brier),
    "odds_ratios": {f: float(np.exp(c)) for f, c in zip(cis.FATIGUE_FEATURES, fat.coef_[0])},
    "bands": {b: {"share": float(r.share), "predicted": float(r.predicted),
                  "actual": float(r.actual_brush_off), "customers": int(r.customers)}
              for b, r in band_table.iterrows()},
    "flagged_at_cutoff": flag_table,
    "propensity_auc": float(prop_auc), "base_rate": float(base),
    "top_decile_rate": float(lift.iloc[0]), "top_decile_lift": float(lift.iloc[0] / base),
    "decision": {a: {"customers": int(r.customers), "subscribe_rate": float(r.subscribe_rate),
                     "brush_off_rate": float(r.brush_off_rate)} for a, r in decision.iterrows()},
}
cis.export_params(fat, prop, high_cut, metrics)
print("\nSaved outputs/model_params.json and 4 charts in outputs/")

In [ ]:
!python train.py

## 4. Charts

In [ ]:
from IPython.display import Image, display
for f in ["fatigue_curve", "roc_curves", "propensity_lift", "decision_outcomes"]:
    display(Image(f"outputs/{f}.png", width=700))

## 5. Six worked examples (`score_demo.py`)

In [ ]:
%%writefile score_demo.py
"""
Score the six worked examples with the saved model (outputs/model_params.json).
Run after train.py:  python score_demo.py
"""
import json

import cis_core as cis

with open("outputs/model_params.json") as f:
    params = json.load(f)

print(f"{'Customer':34s} {'Calls':>5s} {'z':>7s} {'Fatigue':>8s} {'Band':>7s} {'Propensity':>11s}  Action")
for name, c in cis.EXAMPLES.items():
    r = cis.score_from_params(params, c)
    print(f"{name:34s} {c['calls_this_campaign']:5d} {r['fatigue_z']:7.3f} {r['fatigue']:8.1f} "
          f"{r['band']:>7s} {r['propensity']:11.1%}  {r['action']}")
print(f"\nHigh propensity means >= {params['propensity']['high_cut']:.1%} (top 30% of customers).")
print("Tired means fatigue score > 30 (Medium or High band).")

In [ ]:
!python score_demo.py

## 6. Score your own customer
Change any value below and run the cell again.

In [ ]:
import json, cis_core as cis
params = json.load(open("outputs/model_params.json"))
customer = dict(
    calls_this_campaign=4,      # calls made in this campaign
    earlier_contacts=0,         # contacts in earlier campaigns
    contact="cellular",         # "cellular" or "telephone"
    age=40, job="technician", marital="married", education="university.degree",
    default="no", poutcome="nonexistent", month="may", euribor3m=4.86,
    opted_out=False,
)
r = cis.score_from_params(params, customer)
print(f"Fatigue score : {r['fatigue']:.1f} / 100  ({r['band']})")
print(f"Propensity    : {r['propensity']:.1%}")
print(f"Decision      : {r['action']}")